In [ ]:

import torch
import pandas as pd
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from typing import List, Tuple
import numpy as np


C:\Users\charlie.burgwardt\AppData\Local\Temp\ipykernel_34044\44375895.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("edgeconv_multi_dwells_sensor_f

In [ ]:

# Load checkpoint
ckpt = torch.load("edgeconv_multi_dwells_sensor_fixed_best.pt", map_location="cpu")

# Extract preprocessing info
base_cols = ckpt["base_cols"]
feature_cols = ckpt["feature_cols"]
mean, std = ckpt["feat_mean"], ckpt["feat_std"]
knn_on = ckpt["knn_on_cols"]
k = ckpt["k"]

# Rebuild model
from gnn_edgeconv_multi_dwells_sensor_fixed import EdgeConvNet
model = EdgeConvNet(in_channels=len(feature_cols),
                    hidden_channels=64, num_layers=2,
                    mlp_hidden=64, dropout=0.2)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()


In [5]:

# Rebuild graphs from new CSV (no labels -> set label to 0 as placeholder)
def build_graphs_for_inference(csv_path, dwell_col="dwell_id", extra_cols=[]):
    df = pd.read_csv(csv_path)
    graphs = []
    for dwell_id, gdf in df.groupby(dwell_col):
        x_sensor = gdf[["x_sensor","y_sensor","z_sensor"]].values.astype("float32")
        x_target = gdf[["x_target","y_target","z_target"]].values.astype("float32")
        x_rel    = x_target - x_sensor
        base     = gdf[base_cols].values.astype("float32")
        extras   = gdf[extra_cols].values.astype("float32") if extra_cols else None
        x_np     = base if extras is None else np.concatenate([base, extras], axis=1)
        x_np     = np.concatenate([x_np, x_rel], axis=1)
        # dummy labels for shape; we'll ignore with valid_mask
        y_np     = np.zeros((x_np.shape[0],), dtype="int64")
        data = Data(x=torch.from_numpy(x_np), y=torch.from_numpy(y_np))
        data.dwell_id = dwell_id
        graphs.append(data)
    return graphs


In [6]:

def add_sensor_node_and_star_edges(g: Data,
                                   feature_cols: List[str],
                                   mean: torch.Tensor,
                                   std: torch.Tensor,
                                   x_std_detection: torch.Tensor,
                                   edge_index_detection: torch.Tensor) -> None:
    """
    Prepend a sensor node to g.x and connect it to all detection nodes (star edges).
    - Sensor node features: zeros except sensor coords set to per-dwell average, then standardized.
    - g.y is extended with a dummy label for sensor and a valid_mask excludes sensor from loss/metrics.
    - Existing detection edges are shifted by +1 and concatenated with star edges.
    """
    # Build a sensor feature vector in raw space (same dimensionality as g.x)
    # Current g.x is raw (before standardization) when this is called
    sensor_idx = [feature_cols.index('x_sensor'), feature_cols.index('y_sensor'), feature_cols.index('z_sensor')]

    # Compute per-dwell average sensor coordinates from raw detection nodes
    # We need access to raw detection features; since we passed x_std_detection (standardized),
    # we can reconstruct raw via mean/std if needed, but simpler: compute from g.x (raw) now.
    x_raw_detection = g.x  # raw features (before standardization)
    sensor_raw = torch.zeros((1, x_raw_detection.size(1)), dtype=x_raw_detection.dtype)
    sensor_raw[0, sensor_idx] = x_raw_detection[:, sensor_idx].mean(dim=0)
    # Standardize sensor features consistently
    sensor_std = (sensor_raw - mean) / std

    # Prepend sensor node features to standardized detection features
    g.x = torch.cat([sensor_std, x_std_detection], dim=0)  # [1 + N, F]

    # Extend y with a dummy label for sensor and create valid mask
    N = x_std_detection.size(0)
    y_sensor = torch.tensor([-1], dtype=g.y.dtype)  # sentinel
    g.y = torch.cat([y_sensor, g.y], dim=0)
    valid_mask = torch.zeros(N + 1, dtype=torch.bool)
    valid_mask[1:] = True  # exclude sensor node at index 0
    g.valid_mask = valid_mask

    # Build star edges sensor<->detections
    src = torch.zeros(N, dtype=torch.long)  # sensor node id 0
    dst = torch.arange(1, N + 1, dtype=torch.long)
    star_edges = torch.vstack([torch.cat([src, dst]), torch.cat([dst, src])])  # bidirectional

    # Shift detection edges by +1 and combine
    if edge_index_detection is not None and edge_index_detection.numel() > 0:
        edge_index_detection = edge_index_detection + 1  # shift node ids
        g.edge_index = torch.cat([edge_index_detection, star_edges], dim=1)
    else:
        g.edge_index = star_edges


In [12]:

# Standardize, build kNN among detections, add sensor node + star edges
def prepare_for_inference(graphs):
    def build_knn_edges(x_std):
        idxs = [feature_cols.index(c) for c in knn_on]
        x_knn = x_std[:, idxs]
        from torch_geometric.nn import knn_graph
        N = x_std.size(0)
        k_eff = max(1, min(k, max(1, N-1)))
        return knn_graph(x_knn, k=k_eff, loop=False) if N > 1 else torch.empty((2,0), dtype=torch.long)

    for g in graphs:
        x_std = (g.x - mean) / std
        edge_det = build_knn_edges(x_std)
        # Add sensor node and star edges exactly like training
        # (reuse your add_sensor_node_and_star_edges function)
        # For brevity, assume you imported it from your script
        add_sensor_node_and_star_edges(g, feature_cols, mean, std, x_std, edge_det)


In [13]:

# Run inference and save predictions per dwell
def run_inference(model, graphs, out_csv="predictions.csv", device="cpu"):
    model.eval()
    rows = []
    for g in graphs:
        x = g.x.to(device)
        edge_index = g.edge_index.to(device)
        with torch.no_grad():
            logits = model(x, edge_index)
            probs = torch.sigmoid(logits[g.valid_mask]).cpu().numpy()
        for i, p in enumerate(probs):
            rows.append({"dwell_id": getattr(g, "dwell_id", "unknown"),
                         "node_idx": i, "prob": float(p), "pred": int(p >= 0.5)})
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"Saved predictions to {out_csv}")


In [17]:
csv_path = 'C:/Users/charlie.burgwardt/OneDrive - NV5/GMTI/Data/all_dwells.csv'

graphs = build_graphs_for_inference(csv_path, dwell_col="dwell_id", extra_cols=[])

prepare_for_inference(graphs)

run_inference(model, graphs, out_csv="predictions.csv", device="cpu")



AttributeError: 'dict' object has no attribute 'eval'